In [4]:
# ============================================================
# EXPERIMENT 3 - DATA PREPROCESSING
# WORLD HAPPINESS DATASET 2015-2024
# ============================================================


# ============================================================
# 1. IMPORT DATASET DIRECTLY FROM KAGGLE
# ============================================================

!pip install -q kagglehub

import kagglehub
import pandas as pd
import glob
import os

# Download directly from Kaggle
path = kagglehub.dataset_download(
    "yadiraespinoza/world-happiness-2015-2024"
)

print("Dataset downloaded successfully!")
print("Path:", path)


# ============================================================
# 2. FIND AND READ CSV FILES
# ============================================================

csv_files = glob.glob(
    os.path.join(path, "**", "*.csv"),
    recursive=True
)

print("\nCSV Files:")
for file in csv_files:
    print(os.path.basename(file))


# Read all CSV files
dataframes = []

for file in csv_files:
    temp = pd.read_csv(file, delimiter=';')

    # Store source/year
    temp["Year_File"] = os.path.basename(file)

    dataframes.append(temp)


# Combine all years
df = pd.concat(
    dataframes,
    ignore_index=True
)


print("\nFirst 5 rows:")
print(df.head())

print("\nShape:")
print(df.shape)


# ============================================================
# 2.1. Convert numeric columns with comma decimals to float
# ============================================================

for col in df.columns:
    # Check if the column is of object type (likely strings)
    if df[col].dtype == 'object':
        # Check if any value in the column contains a comma
        if df[col].astype(str).str.contains(',').any():
            try:
                # Replace comma with dot and convert to numeric
                df[col] = df[col].astype(str).str.replace(',', '.', regex=False).astype(float)
            except ValueError:
                # If conversion fails, it's not a numeric column with comma decimals
                pass


# ============================================================
# 3. CHECK MISSING VALUES
# ============================================================

print("\nMissing Values:")
print(df.isnull())

print("\nTotal Missing Values:")
print(df.isnull().sum())


# ============================================================
# 4. HANDLE MISSING VALUES
# ============================================================

# Find categorical columns
categorical_columns = df.select_dtypes(
    include=["object"]
).columns

# Fill categorical missing values with mode
for col in categorical_columns:

    if df[col].isnull().sum() > 0:

        df[col] = df[col].fillna(
            df[col].mode()[0]
        )


# Find numerical columns
numerical_columns = df.select_dtypes(
    include=["int64", "float64"]
).columns

# Fill numerical missing values with median
for col in numerical_columns:

    if df[col].isnull().sum() > 0:

        df[col] = df[col].fillna(
            df[col].median()
        )


print("\nMissing values after preprocessing:")
print(df.isnull().sum())


# ============================================================
# 5. CHECK DUPLICATES
# ============================================================

print("\nDuplicate Records:")
print(df.duplicated())

print("\nNumber of duplicate records:")
print(df.duplicated().sum())


# Remove duplicates
df.drop_duplicates(
    inplace=True
)


print("\nDataset after removing duplicates:")
print(df)


# ============================================================
# 6. OUTLIER DETECTION USING IQR
# ============================================================

# Find a happiness score column

if "Life Ladder" in df.columns:

    score_column = "Life Ladder"

elif "Ladder score" in df.columns:

    score_column = "Ladder score"

elif "Happiness Score" in df.columns:

    score_column = "Happiness Score"

else:

    score_column = None


# Detect outliers
if score_column is not None:

    Q1 = df[score_column].quantile(0.25)

    Q3 = df[score_column].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR

    upper = Q3 + 1.5 * IQR

    outliers = df[
        (df[score_column] < lower) |
        (df[score_column] > upper)
    ]

    print("\nOutliers in", score_column, ":")

    print(outliers)


# ============================================================
# 7. NORMALIZATION USING MINMAXSCALER
# ============================================================

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()


# Select numerical columns
num_cols = df.select_dtypes(
    include=["int64", "float64"]
).columns


# Apply Min-Max scaling
df[num_cols] = scaler.fit_transform(
    df[num_cols]
)


print("\nNormalized Numerical Columns:")
print(df[num_cols].head())


# ============================================================
# 8. LABEL ENCODING
# ============================================================

from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()


# Encode Country column
if "Country name" in df.columns:

    df["Country name"] = encoder.fit_transform(
        df["Country name"].astype(str)
    )

elif "Country" in df.columns:

    df["Country"] = encoder.fit_transform(
        df["Country"].astype(str)
    )


print("\nAfter Label Encoding:")
print(df.head())


# ============================================================
# 9. ONE-HOT ENCODING
# ============================================================

# Encode Year_File using One-Hot Encoding

one_hot = pd.get_dummies(
    df["Year_File"],
    prefix="Year"
)

print("\nOne-Hot Encoded Year:")
print(one_hot.head())


# Add one-hot columns to dataset
df = pd.concat(
    [df, one_hot],
    axis=1
)


# Remove original Year_File column
df.drop(
    "Year_File",
    axis=1,
    inplace=True
)


# ============================================================
# 10. FINAL PREPROCESSED DATASET
# ============================================================

print("\n===================================")
print("FINAL PREPROCESSED DATASET")
print("===================================")

print(df.head())

print("\nFinal Shape:")
print(df.shape)

print("\nRemaining Missing Values:")
print(df.isnull().sum())

print("\nFinal Dataset Information:")
df.info()

Using Colab cache for faster access to the 'world-happiness-2015-2024' dataset.
Dataset downloaded successfully!
Path: /kaggle/input/world-happiness-2015-2024

CSV Files:
world_happiness_2018.csv
world_happiness_2023.csv
world_happiness_2022.csv
world_happiness_2016.csv
world_happiness_2015.csv
world_happiness_2017.csv
world_happiness_2019.csv
world_happiness_2020.csv
world_happiness_2021.csv
world_happiness_2024.csv
world_happiness_combined.csv

First 5 rows:
   Ranking      Country            Regional indicator Happiness score  \
0      145  Afghanistan                    South Asia          3,6315   
1      112      Albania    Central and Eastern Europe           4,586   
2       84      Argelia  Middle East and North Africa          5,2946   
3      142       Angola            Sub-Saharan Africa          3,7948   
4       29    Argentina   Latin America and Caribbean           6,388   

  GDP per capita Social support  Healthy life expectancy  \
0        2,01215        0,20121     